# PARADE UTR Prediction & Design

![PARADE UTR Prediction & Design](https://proto-bio.github.io/proto-assets/images/tool/parade/hero.png)

PARADE (Prediction And RAtional DEsign of mRNA UTRs; [Khoroshkin et al., 2024](https://doi.org/10.1101/2024.12.31.630783)) is a LegNet convolutional model for the untranslated regions of mRNA. This toolkit wraps it as four tools: `parade-activity` (cell-type-specific 5'/3' UTR activity), `parade-stability` (3' UTR mRNA stability as an RNA/gDNA log-ratio), `parade-gradient` (differentiable per-cell UTR activity and its gradient for gradient-based design), and `parade-stability-gradient` (the differentiable form of the stability prediction).

This notebook scores UTRs with each tool, then uses `parade-gradient` in a short Fast SeqProp loop to *design* UTRs whose predicted activity is specific to one cell line over another. PARADE uses anonymized cell-line codes: the 5'UTR panel is `c1, c2, c4, c6, c17` and the 3'UTR panel adds `c13` (`c2` is HepG2). The published checkpoints download automatically on first use from a pinned `autosome-ru/parade` commit into the managed model cache; a GPU speeds things up but everything below also runs on CPU (set `DEVICE = "cpu"`).

In [ ]:
from proto_tools.utils.notebook_docs import (
    display_api_reference,
    display_available_tools,
    display_doc_link,
    display_docs_section,
    display_overview,
)

display_doc_link("parade")                    # link to the hosted docs page
display_overview("parade")                    # one-paragraph summary
display_docs_section("parade", "Background")  # scientific background from the toolkit README

## Available tools

In [ ]:
display_available_tools("parade")

### `run_parade_activity`

Predicts an activity value **per requested cell code** for each UTR. Set `construct_type` to `utr5` or `utr3` (it selects the checkpoint and cell-code panel); leave `cell_types` empty to score the full panel. Upstream trained the 5'UTR model on ~50-nt inserts and the 3'UTR model on ~240-nt inserts, so score near those lengths. Mixed lengths in one call are batched per length group, and RNA input (`U`) is mapped to DNA (`T`).

In [ ]:
from proto_tools.tools import ParadeActivityConfig, ParadeActivityInput, run_parade_activity

# The tool runs the model inside its own isolated environment; DEVICE only selects that worker's
# device. Use "cpu" if you don't have a CUDA GPU.
DEVICE = "cuda"

In [ ]:
display_api_reference("parade-activity", "input", "run_parade_activity")

# Three 50-nt 5'UTR inserts to score across the full 5'UTR panel.
utrs = [
    "GCACCATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATC",  # GC-rich insert
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA",  # low-complexity control
    "GCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGC",
]
inputs = ParadeActivityInput(sequences=utrs)

In [ ]:
display_api_reference("parade-activity", "config", "run_parade_activity")

# cell_types left empty -> the full 5'UTR panel (c1, c2, c4, c6, c17).
config = ParadeActivityConfig(construct_type="utr5", device=DEVICE)

In [ ]:
activity = run_parade_activity(inputs, config)

In [ ]:
display_api_reference("parade-activity", "output", "run_parade_activity")

# `activity.results` is one item per input; `result.scores` is a metrics container keyed by cell code.
cols = activity.cell_types
print("sequence           | " + " | ".join(f"{c:>7}" for c in cols))
for r in activity.results:
    row = dict(r.scores.items())
    print(f"{r.sequence[:16]:<18} | " + " | ".join(f"{row[c]:7.3f}" for c in cols))

### `run_parade_stability`

Predicts 3' UTR mRNA stability as an RNA/gDNA **log-ratio** — higher is more stable, one value per sequence (there is no cell conditioning). Upstream trained this model on 186-nt sequences, so score near that length.

In [ ]:
from proto_tools.tools import ParadeStabilityConfig, ParadeStabilityInput, run_parade_stability

In [ ]:
display_api_reference("parade-stability", "input", "run_parade_stability")

# The stability model was trained on 186-nt 3'UTR sequences; score near that length.
stability_inputs = ParadeStabilityInput(sequences=["ACGT" * 46 + "AC", "TGCA" * 46 + "GT"])  # 186 nt

In [ ]:
display_api_reference("parade-stability", "config", "run_parade_stability")

stability_config = ParadeStabilityConfig(device=DEVICE)

In [ ]:
stability = run_parade_stability(stability_inputs, stability_config)

In [ ]:
display_api_reference("parade-stability", "output", "run_parade_stability")

for r in stability.results:
    print(f"{r.sequence[:16]}...  log_ratio = {r.log_ratio:+.3f}")

### `run_parade_gradient`

The design primitive. Instead of a discrete sequence it takes **relaxed logits** of shape `(batch, length, 4)` — a soft, differentiable stand-in for a one-hot sequence — and returns, for each requested cell, that cell's raw predicted activity and its own **gradient** with respect to those logits. Each `ParadeActivityTerm` in `gradient_terms` names a `cell_type`; the goals are applied afterwards with `result.compose(...)`, so "maximize `c2`, minimize `c6`" is a cell-type-specificity objective.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from proto_tools.tools import (
    ParadeActivityTerm,
    ParadeGradientConfig,
    ParadeGradientInput,
    run_parade_gradient,
)

# parade-gradient works on relaxed nucleotide logits with columns in A,C,G,T order.
VOCAB = "ACGT"

In [ ]:
display_api_reference("parade-gradient", "input", "run_parade_gradient")

# One random candidate: relaxed logits of shape (batch=1, length=50, 4).
rng = np.random.default_rng(0)
gradient_inputs = ParadeGradientInput(inputs=rng.normal(0, 1, size=(1, 50, 4)).tolist(), temperature=1.0)

In [ ]:
display_api_reference("parade-gradient", "config", "run_parade_gradient")

# Differentiate c2 and c6 activity; the goals that make this a cell-type-specificity
# objective are applied afterwards, when the terms are composed.
gradient_config = ParadeGradientConfig(
    construct_type="utr5",
    gradient_terms=[
        ParadeActivityTerm(cell_type="c2"),
        ParadeActivityTerm(cell_type="c6"),
    ],
    device=DEVICE,
)

In [ ]:
gradient = run_parade_gradient(gradient_inputs, gradient_config)

In [ ]:
display_api_reference("parade-gradient", "output", "run_parade_gradient")

candidate = gradient.results[0]                        # one result per design
c2_term, c6_term = candidate.terms                     # one term per requested cell
# Compose the raw terms into one minimizable objective: push c2 up, c6 down.
objective = candidate.compose({c2_term.term: "maximize", c6_term.term: "minimize"})
g = np.array(objective.gradient)                       # (50, 4): dLoss/dlogits
print("objective loss :", round(objective.objective_loss, 4))
print("gradient shape :", g.shape)
print("raw c2 activity:", round(c2_term.term_value, 3), "| raw c6 activity:", round(c6_term.term_value, 3))

### `run_parade_stability_gradient`

`run_parade_gradient` above differentiates **activity** — the per-cell UTR activity the activity checkpoints predict. This tool differentiates **stability**: the RNA/gDNA log-ratio from the separate stability checkpoint, where higher means more stable.

Because stability is a single scalar per sequence rather than a per-cell panel, there is nothing to select: each result carries one `log_ratio` term with its raw value and its gradient, and which way to step is a goal the caller applies afterwards when composing that term into an objective. Pair it with the activity gradient when a design needs both an expression profile and a stability target, treating the two as separate objectives.

In [ ]:
from proto_tools.tools import (
    ParadeStabilityGradientConfig, ParadeStabilityGradientInput, run_parade_stability_gradient,
)

display_api_reference("parade-stability-gradient", "input", "run_parade_stability_gradient")

# Relax the same two 186-nt 3'UTRs scored above: a sharpened one-hot of each real sequence.
# Every candidate in a call must share a length -- the backend runs one vectorized pass.
stab_grad_sharpness = 2.0
stab_grad_logits = [
    [[stab_grad_sharpness if base == observed else 0.0 for base in "ACGT"] for observed in seq.upper()]
    for seq in stability_inputs.sequences
]
print(f"{len(stab_grad_logits)} candidates x {len(stab_grad_logits[0])} nt")


In [ ]:
display_api_reference("parade-stability-gradient", "config", "run_parade_stability_gradient")

# Ask how each base change moves the predicted stability of each UTR.
stab_grad_output = run_parade_stability_gradient(
    ParadeStabilityGradientInput(inputs=stab_grad_logits),
    ParadeStabilityGradientConfig(),
)


In [ ]:
display_api_reference("parade-stability-gradient", "output", "run_parade_stability_gradient")

# One result per candidate, in input order, each carrying a single log_ratio term with the raw
# predicted log-ratio and its gradient.
for stab_grad_seq, stab_grad_item in zip(stability_inputs.sequences, stab_grad_output.results):
    stab_grad_term = stab_grad_item.terms[0]
    print(f"{stab_grad_seq[:16]}...  log_ratio = {stab_grad_term.term_value:+.3f}  "
          f"length = {stab_grad_item.metrics['sequence_length']}  gradient = "
          f"({len(stab_grad_term.gradient)}, {len(stab_grad_term.gradient[0])})")

# These log-ratios differ from the discrete scores printed by run_parade_stability above, and
# they should: at sharpness 2.0 the model sees softmax([2, 0, 0, 0]) per position -- roughly
# [0.47, 0.18, 0.18, 0.18] -- a blurred sequence, not the wild-type base. Forwarding a hard
# one-hot instead (hard=1.0, the straight-through estimator) recovers the discrete prediction
# while keeping a differentiable backward path.
stab_grad_hard = run_parade_stability_gradient(
    ParadeStabilityGradientInput(inputs=stab_grad_logits),
    ParadeStabilityGradientConfig(hard=1.0),
)
print()
for stab_grad_seq, stab_grad_item in zip(stability_inputs.sequences, stab_grad_hard.results):
    print(f"{stab_grad_seq[:16]}...  hard-forward log_ratio = {stab_grad_item.terms[0].term_value:+.3f}")


## Design cell-type-specific UTRs (Fast SeqProp)

We turn the gradient into a design loop, [Fast SeqProp](https://doi.org/10.1186/s12859-021-04437-5)-style: start from random relaxed logits and repeatedly (1) ask `parade-gradient` for the gradient of the specificity objective, then (2) take an **Adam** step on the logits. Descending the loss raises on-target activity and lowers off-target activity, so specificity `activity(on) - activity(off)` climbs; at the end we `argmax` the logits into a concrete DNA sequence.

In [ ]:
def decode(logits):
    """Decode (N, L, 4) relaxed logits to DNA strings by per-position argmax (A,C,G,T order)."""
    return ["".join(VOCAB[i] for i in row.argmax(-1)) for row in logits]


def design_specific_utrs(
    construct_type, on_target, off_target,
    seq_len=50, n_designs=8, steps=50, lr=0.5, seed=0, device=DEVICE, log_every=10,
):
    """Fast SeqProp design of UTRs whose activity is specific to `on_target` over `off_target`.

    Optimizes a batch of `n_designs` relaxed logit tensors in parallel with Adam, driving the
    parade-gradient specificity objective. Returns (history, designs):
      - history: mean (on - off) predicted activity across the batch, per step (the climb);
      - designs: the final decoded DNA sequences.
    """
    # Start near a uniform distribution over nucleotides (small logits -> flat softmax), so the
    # optimizer isn't biased toward any starting sequence.
    rng = np.random.default_rng(seed)
    logits = 0.1 * rng.standard_normal((n_designs, seq_len, 4))

    # Adam optimizer state (first/second moment estimates) and hyperparameters.
    m = np.zeros_like(logits)
    v = np.zeros_like(logits)
    b1, b2, eps = 0.9, 0.999, 1e-8

    # The specificity objective is fixed across steps: push on_target up, off_target down.
    on_term = ParadeActivityTerm(cell_type=on_target)
    off_term = ParadeActivityTerm(cell_type=off_target)
    config = ParadeGradientConfig(
        construct_type=construct_type,
        gradient_terms=[on_term, off_term],
        device=device,
    )
    goals = {on_term: "maximize", off_term: "minimize"}

    history = []
    for t in range(1, steps + 1):
        # 1) One differentiable evaluation of the whole batch. temperature=1.0 is a plain softmax
        #    relaxation of the logits into nucleotide probabilities.
        out = run_parade_gradient(ParadeGradientInput(inputs=logits.tolist(), temperature=1.0), config)

        # 2) Compose each design's two terms under the goals above into one minimizable objective,
        #    then read its gradient w.r.t. the logits, same shape as `logits`.
        objectives = [r.compose(goals) for r in out.results]
        grad = np.array([o.gradient for o in objectives])  # (n_designs, L, 4)

        # 3) Track the mean specificity for this step. `results[i].terms` carries the raw activity
        #    of each objective cell for design i, so specificity_i = activity(on) - activity(off).
        spec = float(np.mean([r.terms[0].term_value - r.terms[1].term_value for r in out.results]))
        history.append(spec)

        # 4) Adam update. We DESCEND the loss (subtract the step); because loss falls as on-target
        #    activity rises and off-target falls, the specificity we track goes up.
        m = b1 * m + (1 - b1) * grad
        v = b2 * v + (1 - b2) * grad ** 2
        m_hat = m / (1 - b1 ** t)
        v_hat = v / (1 - b2 ** t)
        logits -= lr * m_hat / (np.sqrt(v_hat) + eps)

        if log_every and (t == 1 or t % log_every == 0):
            print(f"  step {t:3d} | mean {on_target}-{off_target} specificity = {spec:+.3f}")

    # Collapse the optimized soft logits to concrete sequences.
    return history, decode(logits)

### Design 5'UTRs specific to `c2` over `c6`

In [ ]:
hist_utr5, designs_utr5 = design_specific_utrs("utr5", on_target="c2", off_target="c6")
print("\nexample designed 5'UTR:", designs_utr5[0])

### Design 3'UTRs specific to `c2` over `c13`

In [ ]:
hist_utr3, designs_utr3 = design_specific_utrs("utr3", on_target="c2", off_target="c13", seq_len=240)  # 3'UTR training length
print("\nexample designed 3'UTR:", designs_utr3[0])

### The climb

Mean predicted specificity across the 8 designs, per Fast SeqProp step — both panels climb from ~0.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
# One line per construct type; markers every step so the trajectory is easy to read.
ax.plot(range(1, len(hist_utr5) + 1), hist_utr5, color="#2a7de1", marker="o", ms=3, lw=2, label="5'UTR  (c2 vs c6)")
ax.plot(range(1, len(hist_utr3) + 1), hist_utr3, color="#e07b39", marker="o", ms=3, lw=2, label="3'UTR  (c2 vs c13)")
ax.axhline(0.0, color="0.6", lw=1, ls="--", zorder=0)  # zero specificity reference
ax.set_xlabel("Fast SeqProp step")
ax.set_ylabel("mean predicted specificity  (on \u2212 off activity)")
ax.set_title("PARADE-guided UTR design: cell-type specificity climbs")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### Verify the designs in discrete space

The loop optimizes *relaxed* logits; what ultimately matters is the discrete `argmax` sequence. We re-score the decoded designs with `parade-activity` and compare their specificity to random sequences of the same length — the designs should be far more specific.

In [ ]:
def discrete_specificity(construct_type, seqs, on, off):
    """Mean (on - off) activity for a list of concrete sequences, via parade-activity."""
    out = run_parade_activity(
        ParadeActivityInput(sequences=seqs),
        ParadeActivityConfig(construct_type=construct_type, cell_types=[on, off], device=DEVICE),
    )
    return float(np.mean([r.scores[on] - r.scores[off] for r in out.results]))


# Random-sequence baselines of the same length, for reference.
rng = np.random.default_rng(1)
random_utr5 = ["".join(rng.choice(list(VOCAB), 50)) for _ in range(8)]
random_utr3 = ["".join(rng.choice(list(VOCAB), 240)) for _ in range(8)]  # match the 240-nt designs

rows = [
    ("5'UTR c2-c6", discrete_specificity("utr5", designs_utr5, "c2", "c6"),
                    discrete_specificity("utr5", random_utr5, "c2", "c6")),
    ("3'UTR c2-c13", discrete_specificity("utr3", designs_utr3, "c2", "c13"),
                     discrete_specificity("utr3", random_utr3, "c2", "c13")),
]
header = ("panel", "designed", "random")
print(f"{header[0]:<16}{header[1]:>10}{header[2]:>10}")
for panel, designed, random_ in rows:
    print(f"{panel:<16}{designed:>10.3f}{random_:>10.3f}")

## Export Results

Every scorer output exports to its supported formats (`json`, `csv`).

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

activity.export(name="parade_activity", export_path=output_dir, file_format="csv")
stability.export(name="parade_stability", export_path=output_dir, file_format="json")
print(f"Results exported to {output_dir}/")